# strat_42: our final submission, rebuilt from the checkpoint

This notebook reproduces the file we submitted as our final model (`strat_42`, public 0.910, private 0.878).
It does no training. It loads the saved weights and

1. builds the network and prints its parameter count (limit 5,000,000),
2. prints the validation accuracy of the recipe on VAL5K,
3. rebuilds `submission.csv` from `outputs_s0/model.pt` and checks it against the submitted file,
4. lists the Kaggle scores of the submitted file.

strat_42 is the strat_40 recipe (ResNet 3-3-3, RegMixup + SLA+SD, 400 epochs) trained on all 50,000 images, so
the model itself has no validation score: VAL5K was part of its training data. Step 2 therefore evaluates the
same recipe trained without VAL5K (strat_40, seed 0) with the same inference (EMA weights, class head, 6-view TTA).

It runs on a CPU in a few minutes. You need `packed/cifar_comp_v1.npz` (see the top-level README).

In [1]:
import hashlib
import sys
from pathlib import Path

import numpy as np
import torch

HERE = Path.cwd()                       # run from strategies/strat_42_rm400_sla_full
REPO = HERE.parents[1]
DATA = REPO / "packed" / "cifar_comp_v1.npz"
S40 = REPO / "strategies" / "strat_40_rm400_sla"
sys.path.insert(0, str(HERE))
import strat_42_train as T              # same data loading, preprocessing and TTA code as training

cfg, cfg_sha = T.load_cfg(HERE / "strat_42_config.yaml")
dev, gpu, _, _, _ = T.setup_device(cfg)
print("torch", torch.__version__, "| device:", gpu)
print("config sha256:", cfg_sha[:16])
print("data file found:", DATA.exists())

torch 2.9.1+cpu | device: cpu
config sha256: 82deefee97e91251
data file found: True


## 1. Parameter count

Every parameter is counted, including the 40-way rotation head that SLA+SD trains.

In [2]:
model = T.build_model(cfg["model"])
by_part = {}
for name, p in model.named_parameters():
    part = name.split(".")[0]
    by_part[part] = by_part.get(part, 0) + p.numel()
total = sum(p.numel() for p in model.parameters())
for part, n in by_part.items():
    print(f"{part:>12}: {n:>10,}")
print(f"{'total':>12}: {total:>10,}  (limit 5,000,000)")
assert total <= 5_000_000
assert total == cfg["expected_params"]

        stem:      1,856
      layers:  4,323,328
          fc:      2,570
    fc_joint:     10,280
       total:  4,338,034  (limit 5,000,000)


## 2. Validation accuracy of the recipe (strat_40 seed 0 on VAL5K)

The README numbers (98.12 flip2, 98.06 tta6) come from the GPU run. On a CPU in float32 a value can move by an
image or two (0.02 points per image).

In [3]:
D = T.load_data(DATA, cfg, False, dev, need_train=False)
mean, std = T.norm_consts(cfg, dev)
print("VAL5K images:", D["n_val"], "| test images:", len(D["test_ids"]))

def load(path):
    m = T.build_model(cfg["model"])
    m.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    return m.to(memory_format=torch.channels_last).to(dev).eval()

m40 = load(S40 / "outputs_s0" / "model.pt")
probs = T.option_probs(T.predict_logits(m40, D["val_x"], T.TTA6_ORDER, cfg, mean, std), cfg)
for opt in ("none", "flip2", "tta6"):
    print(f"strat_40 seed 0, EMA, class head, {opt:>5}: VAL5K {100 * T.accuracy(probs[opt], D['val_y']):.2f}%")

VAL5K images: 5000 | test images: 2000


strat_40 seed 0, EMA, class head,  none: VAL5K 97.88%
strat_40 seed 0, EMA, class head, flip2: VAL5K 98.12%
strat_40 seed 0, EMA, class head,  tta6: VAL5K 98.04%


## 3. Rebuild the submitted file from the strat_42 checkpoint

In [4]:
def sha256(path, text=False):
    b = Path(path).read_bytes()
    if text:  # LF line endings, as stored in git (a Windows checkout may turn them into CRLF)
        b = b.replace(b"\r\n", b"\n")
    return hashlib.sha256(b).hexdigest()

ckpt = HERE / "outputs_s0" / "model.pt"
print("checkpoint sha256:", sha256(ckpt)[:16])
m42 = load(ckpt)
p_test = T.option_probs(T.predict_logits(m42, D["test_x"], T.TTA6_ORDER, cfg, mean, std), cfg)["tta6"]
pred = p_test.argmax(1).cpu().numpy()

out = HERE / "repro"
out.mkdir(exist_ok=True)
T.write_submission(out / "submission.csv", D["test_ids"], pred)

submitted = HERE / "outputs_s0" / "submission.csv"
old = np.loadtxt(submitted, delimiter=",", skiprows=1, dtype=np.int64)[:, 1]
print("rows:", len(pred), "| labels differing from the submitted file:", int((old != pred).sum()))
print("rebuilt   sha256:", sha256(out / "submission.csv", text=True)[:16])
print("submitted sha256:", sha256(submitted, text=True)[:16])
print("predicted class counts:", np.bincount(pred, minlength=10).tolist())

checkpoint sha256: 529685e8d1e4f9da


rows: 2000 | labels differing from the submitted file: 0
rebuilt   sha256: 642a6a786d549d0f
submitted sha256: 642a6a786d549d0f
predicted class counts: [176, 219, 211, 202, 197, 209, 180, 206, 204, 196]


## 4. Kaggle scores of the submitted file

Kaggle scores the 2,000 test images in two fixed halves of 1,000. These numbers come from Kaggle, not from this
notebook (we never had the test labels).

In [5]:
kaggle = {"file": "strat_42_s0_fullfit_tta6.csv", "public": 0.910, "private": 0.878}
print(f"final model: {total:,} parameters | public accuracy {kaggle['public']:.3f} | private accuracy {kaggle['private']:.3f}")

final model: 4,338,034 parameters | public accuracy 0.910 | private accuracy 0.878
